# Name That LLM: smoke de velocidade do LoRA24 (Qwen3-8B-Base, T4 x2)

Mede tokens/s, memória de pico e passos com gradiente não finito de um treino LoRA nas primeiras 24 camadas do
Qwen3-8B-Base, com a arquitetura da receita pública (LoRA r16, leitura em 8 profundidades, cabeça linear).
Um worker por GPU roda ao mesmo tempo. Não avalia acurácia; serve para decidir se vale gastar a cota de GPU
no treino de verdade. Entradas: modelo `qwen-lm/qwen-3/Transformers/8b-base/1` e dataset `crtproduction/name-that-llm-open-pool`. Internet desligada.
Código: `scripts/smoke_lora24.py`.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

SCRIPT = '"""Smoke de velocidade do LoRA24: primeiras 24 camadas de um Qwen3, leitura em várias profundidades, em T4.\n\nMede tokens/s, memória de pico e passos com perda ou gradiente não finito, por GPU e por tamanho de micro-lote,\ncom a arquitetura da receita pública (LoRA r16 em todas as lineares, 8 profundidades com LayerNorm, cabeça linear,\ngradient checkpointing). Não avalia acurácia: serve para decidir se vale gastar a cota de GPU no treino de verdade.\n\nUm worker por GPU roda ao mesmo tempo, o que também testa o treino em duas T4 (cada uma com sua cópia das camadas).\nLocal, com um modelo minúsculo: SMOKE_CFG=\'{"base": "...", "tokenizer": "...", "pool": "..."}\' python scripts/smoke_lora24.py\nNo Kaggle: notebook gerado por scripts/montar_notebook_smoke_lora24.py (internet desligada).\n"""\n\nimport json\nimport math\nimport os\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\n\nCFG = {\n    "base": "",  # pasta do modelo; vazio = procurar em /kaggle/input pelo base_spec\n    "base_spec": {"model_type": "qwen3", "hidden_size": 4096, "num_hidden_layers": 36},\n    "tokenizer": "",  # vazio = o da base\n    "keep": 24,  # camadas mantidas\n    "layers": [2, 4, 6, 8, 12, 16, 20, 24],  # profundidades lidas (hidden_states[l])\n    "r": 16,\n    "max_len": 384,\n    "micro_lotes": [4, 8, 16],\n    "passos": 20,  # passos medidos por tamanho de micro-lote\n    "aquecimento": 3,  # passos antes de começar a cronometrar\n    "n_textos": 2000,\n    "pool": "",  # vazio = procurar ntl_open_pool.parquet em /kaggle/input\n    "tempo_maximo_s": 900,  # teto do treino de cada worker (fora o carregamento)\n    "saida": "/kaggle/working",\n    "gpus": 0,  # 0 = todas as visíveis\n    "textos_projecao": [35000, 223840],\n}\nCFG_JSON = "{}"  # o gerador do notebook troca esta linha\nCFG.update(json.loads(os.environ.get("SMOKE_CFG", CFG_JSON)))\n\nT0 = time.time()\nGPU_TAG = os.environ.get("SMOKE_GPU")  # definido só nos workers\n\n\ndef log(*args):\n    rotulo = "" if GPU_TAG is None else f" gpu{GPU_TAG}"\n    print(f"[{time.time() - T0:6.0f}s{rotulo}]", *args, flush=True)\n\n\ndef preparar_ambiente():\n    """O peft do Kaggle exige torchao >= 0.16 e a imagem traz o 0.10; sem o torchao o peft simplesmente não o usa."""\n    import importlib.metadata as md\n\n    try:\n        versao = md.version("torchao")\n    except md.PackageNotFoundError:\n        return\n    if tuple(int(x) for x in versao.split(".")[:2]) < (0, 16):\n        log(f"torchao {versao} incompatível com o peft: removendo")\n        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)\n\n\ndef achar_base() -> str:\n    if CFG["base"]:\n        return CFG["base"]\n    spec = CFG["base_spec"]\n    for cfg in sorted(Path("/kaggle/input").rglob("config.json")):\n        try:\n            c = json.loads(cfg.read_text())\n        except Exception:\n            continue\n        if all(c.get(k) == v for k, v in spec.items()):\n            return str(cfg.parent)\n    raise FileNotFoundError(f"modelo {spec} não encontrado em /kaggle/input")\n\n\ndef achar_pool() -> str:\n    return CFG["pool"] or str(next(Path("/kaggle/input").rglob("ntl_open_pool.parquet")))\n\n\ndef carregar_dados(tok):\n    pool = pd.read_parquet(achar_pool(), columns=["text", "model", "n_words"])\n    pool = pool[pool.n_words.between(30, 300)]\n    classes = sorted(pool.model.unique())\n    por_classe = max(1, CFG["n_textos"] // len(classes))\n    partes = [g.sample(min(len(g), por_classe), random_state=0) for _, g in pool.groupby("model")]\n    amostra = pd.concat(partes).sample(frac=1, random_state=0).reset_index(drop=True)\n    bruto = tok([t.strip() for t in amostra.text], add_special_tokens=False)["input_ids"]\n    ids = [x[: CFG["max_len"] - 1] + [tok.eos_token_id] for x in bruto]\n    y = amostra.model.map({c: i for i, c in enumerate(classes)}).to_numpy()\n    return ids, y, classes\n\n\nclass Leitor(nn.Module):\n    """Camadas do Qwen3 com LoRA, média dos tokens em várias profundidades (cada uma com LayerNorm) e cabeça linear."""\n\n    def __init__(self, trunk, layers, hidden, n_classes):\n        super().__init__()\n        self.trunk, self.layers = trunk, layers\n        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in layers])\n        self.head = nn.Linear(hidden * len(layers), n_classes)\n\n    def forward(self, ids, mask):\n        hs = self.trunk(input_ids=ids, attention_mask=mask, output_hidden_states=True, use_cache=False).hidden_states\n        assert len(hs) == CFG["keep"] + 1, f"hidden_states com {len(hs)} entradas, esperado {CFG[\'keep\'] + 1}"\n        m = mask.float().clone()\n        m[:, 0] = 0  # o primeiro token é um "attention sink": fora da média\n        m = m.unsqueeze(-1)\n        den = m.sum(1).clamp(min=1)\n        z = torch.cat([nm((hs[l].float() * m).sum(1) / den) for l, nm in zip(self.layers, self.norms)], -1)\n        return self.head(z)\n\n\ndef montar_modelo(base, dispositivo, tipo, n_classes):\n    from peft import LoraConfig, TaskType, get_peft_model\n    from transformers import AutoConfig, AutoModel\n\n    mc = AutoConfig.from_pretrained(base)\n    mc.num_hidden_layers, mc.use_cache = CFG["keep"], False\n    if getattr(mc, "layer_types", None) is not None:\n        mc.layer_types = mc.layer_types[: CFG["keep"]]\n    if dispositivo.startswith("cuda"):\n        trunk = AutoModel.from_pretrained(base, config=mc, dtype=tipo, device_map={"": 0})\n    else:\n        trunk = AutoModel.from_pretrained(base, config=mc, dtype=tipo)\n    trunk.norm = nn.Identity()  # a receita pública lê a camada 24 sem a normalização final\n    trunk.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    trunk.enable_input_require_grads()\n    lora = LoraConfig(task_type=TaskType.FEATURE_EXTRACTION, r=CFG["r"], lora_alpha=2 * CFG["r"], lora_dropout=0.05,\n                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])\n    trunk = get_peft_model(trunk, lora)\n    for p in trunk.parameters():\n        if p.requires_grad:\n            p.data = p.data.float()  # só o adaptador treina, em fp32\n    leitor = Leitor(trunk, CFG["layers"], mc.hidden_size, n_classes)\n    leitor.norms.to(dispositivo)\n    leitor.head.to(dispositivo)\n    treinaveis = sum(p.numel() for p in leitor.parameters() if p.requires_grad)\n    log(f"modelo {base} com {CFG[\'keep\']} camadas em {dispositivo}/{tipo}, treináveis {treinaveis / 1e6:.1f} M")\n    return leitor\n\n\ndef lotes(ids, tamanho, rng):\n    ordem = rng.permutation(len(ids))\n    saida = []\n    for i in range(0, len(ids), tamanho * 50):\n        bloco = sorted(ordem[i : i + tamanho * 50], key=lambda j: len(ids[j]))  # tamanhos parecidos juntos\n        saida += [bloco[k : k + tamanho] for k in range(0, len(bloco), tamanho)]\n    rng.shuffle(saida)\n    return [b for b in saida if len(b) == tamanho] or saida\n\n\ndef montar_lote(ids, indices, pad, dispositivo):\n    maior = max(len(ids[i]) for i in indices)\n    x = torch.full((len(indices), maior), pad, dtype=torch.long)\n    m = torch.zeros((len(indices), maior), dtype=torch.long)\n    for k, i in enumerate(indices):\n        x[k, : len(ids[i])] = torch.tensor(ids[i])\n        m[k, : len(ids[i])] = 1\n    return x.to(dispositivo), m.to(dispositivo)\n\n\ndef sincronizar(dispositivo):\n    if dispositivo.startswith("cuda"):\n        torch.cuda.synchronize()\n\n\ndef medir(leitor, ids, y, mb, dispositivo, tipo, pad, deadline):\n    """Treina `aquecimento + passos` micro-lotes de tamanho mb e cronometra os últimos `passos`."""\n    nucleo = [p for n, p in leitor.named_parameters() if p.requires_grad and n.startswith("trunk")]\n    cabeca = [p for n, p in leitor.named_parameters() if p.requires_grad and not n.startswith("trunk")]\n    todos = nucleo + cabeca\n    otimizador = torch.optim.AdamW([{"params": nucleo, "lr": 1e-4}, {"params": cabeca, "lr": 1e-3}], weight_decay=0.0)\n    cuda = dispositivo.startswith("cuda")\n    escala = torch.amp.GradScaler("cuda", init_scale=1024.0, enabled=(tipo == torch.float16 and cuda))\n    tipo_disp = "cuda" if cuda else "cpu"\n    total = CFG["aquecimento"] + CFG["passos"]\n    fila = lotes(ids, mb, np.random.default_rng(mb))\n    fila = (fila * math.ceil(total / len(fila)))[:total]\n    if cuda:\n        torch.cuda.reset_peak_memory_stats()\n    r = {"micro_lote": mb, "oom": False, "pulados_gradiente": 0, "perda_nao_finita": 0, "passos_medidos": 0}\n    perdas, t_ini, real, preench = [], None, 0, 0\n    leitor.train()\n    for k, indices in enumerate(fila):\n        if time.time() > deadline:\n            log(f"mb={mb}: tempo máximo, parando no passo {k}")\n            break\n        x, m = montar_lote(ids, indices, pad, dispositivo)\n        yb = torch.as_tensor(y[indices], dtype=torch.long, device=dispositivo)\n        if k == CFG["aquecimento"]:\n            sincronizar(dispositivo)\n            t_ini, real, preench = time.time(), 0, 0\n        try:\n            with torch.autocast(device_type=tipo_disp, dtype=tipo, enabled=tipo != torch.float32):\n                logits = leitor(x, m)\n            perda = torch.nn.functional.cross_entropy(logits.float(), yb)\n            if not torch.isfinite(perda):\n                r["perda_nao_finita"] += 1\n                otimizador.zero_grad(set_to_none=True)\n                continue\n            escala.scale(perda).backward()\n            escala.unscale_(otimizador)\n            norma = torch.nn.utils.clip_grad_norm_(todos, 1.0)\n            if torch.isfinite(norma):\n                escala.step(otimizador)\n            else:\n                r["pulados_gradiente"] += 1\n            escala.update()\n            otimizador.zero_grad(set_to_none=True)\n        except torch.cuda.OutOfMemoryError:\n            log(f"mb={mb}: sem memória no passo {k}")\n            r["oom"] = True\n            otimizador.zero_grad(set_to_none=True)\n            del x, m\n            torch.cuda.empty_cache()\n            break\n        perdas.append(float(perda))\n        if k >= CFG["aquecimento"]:\n            real += int(m.sum())\n            preench += x.numel()\n            r["passos_medidos"] += 1\n    sincronizar(dispositivo)\n    if t_ini is not None and r["passos_medidos"]:\n        dt = time.time() - t_ini\n        r.update(s_por_passo=dt / r["passos_medidos"], tokens_reais_por_s=real / dt, tokens_com_pad_por_s=preench / dt)\n    if cuda:\n        r["pico_gb"] = torch.cuda.max_memory_allocated() / 1e9\n    if perdas:\n        r["perda_primeiros3"], r["perda_ultimos3"] = float(np.mean(perdas[:3])), float(np.mean(perdas[-3:]))\n    log("resultado", json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()}))\n    return r\n\n\ndef worker():\n    from transformers import AutoTokenizer\n\n    gpu = int(GPU_TAG)\n    torch.manual_seed(gpu)\n    cuda = torch.cuda.is_available()\n    dispositivo, tipo = ("cuda:0", torch.float16) if cuda else ("cpu", torch.float32)  # T4 não tem bf16 nativo\n    info = {"gpu": gpu, "torch": torch.__version__, "dispositivo": dispositivo}\n    if cuda:\n        info["nome_gpu"] = torch.cuda.get_device_name(0)\n        info["memoria_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)\n    try:\n        import transformers, peft\n\n        info["transformers"], info["peft"] = transformers.__version__, peft.__version__\n    except Exception as e:  # noqa: BLE001\n        info["versoes_erro"] = repr(e)\n    log("ambiente", json.dumps(info))\n    base = achar_base()\n    tok = AutoTokenizer.from_pretrained(CFG["tokenizer"] or base)\n    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id\n    ids, y, classes = carregar_dados(tok)\n    info["tokens_medios_por_texto"] = float(np.mean([len(i) for i in ids]))\n    log(f"{len(ids)} textos, {info[\'tokens_medios_por_texto\']:.0f} tokens em média, {len(classes)} classes")\n    t_carga = time.time()\n    leitor = montar_modelo(base, dispositivo, tipo, len(classes))\n    info["carregamento_s"] = time.time() - t_carga\n    if cuda:\n        info["memoria_apos_carregar_gb"] = torch.cuda.memory_allocated() / 1e9\n    deadline = time.time() + CFG["tempo_maximo_s"]\n    medidas = []\n    for mb in CFG["micro_lotes"]:\n        medidas.append(medir(leitor, ids, y, mb, dispositivo, tipo, pad, deadline))\n        if medidas[-1]["oom"]:\n            break  # tamanhos maiores só pioram\n    (Path(CFG["saida"]) / f"smoke_gpu{gpu}.json").write_text(json.dumps({"info": info, "medidas": medidas}, indent=2))\n    log("worker pronto")\n\n\ndef launcher():\n    preparar_ambiente()\n    saida = Path(CFG["saida"])\n    saida.mkdir(parents=True, exist_ok=True)\n    listagem = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True) if shutil_which("nvidia-smi") else None\n    n_gpus = sum(1 for linha in (listagem.stdout.splitlines() if listagem else []) if linha.startswith("GPU "))\n    if listagem:\n        log(listagem.stdout.strip())\n    n = CFG["gpus"] or max(1, n_gpus)\n    log(f"{n} worker(s), um por GPU" if n_gpus else "sem GPU: 1 worker na CPU (ensaio local)")\n    procs = []\n    for i in range(n):\n        env = dict(os.environ, SMOKE_GPU=str(i), SMOKE_CFG=json.dumps(CFG))\n        if n_gpus:\n            env["CUDA_VISIBLE_DEVICES"] = str(i)\n        procs.append(subprocess.Popen([sys.executable, str(Path(__file__).resolve())], env=env))\n    codigos = [p.wait() for p in procs]\n    log("códigos de saída dos workers", codigos)\n\n    por_gpu = []\n    for i in range(n):\n        arq = saida / f"smoke_gpu{i}.json"\n        if arq.exists():\n            por_gpu.append(json.loads(arq.read_text()))\n    resumo = {"codigos": codigos, "gpus": []}\n    total_tok_s = 0.0\n    tokens_medios = None\n    for g in por_gpu:\n        validas = [m for m in g["medidas"] if m.get("tokens_reais_por_s") and not m["oom"]]\n        melhor = max(validas, key=lambda m: m["tokens_reais_por_s"]) if validas else None\n        tokens_medios = g["info"].get("tokens_medios_por_texto", tokens_medios)\n        resumo["gpus"].append({"info": g["info"], "medidas": g["medidas"], "melhor": melhor})\n        total_tok_s += melhor["tokens_reais_por_s"] if melhor else 0.0\n    resumo["tokens_reais_por_s_total"] = total_tok_s\n    if total_tok_s and tokens_medios:\n        resumo["projecao_horas_1_epoca"] = {\n            str(n_txt): round(n_txt * tokens_medios / total_tok_s / 3600, 4) for n_txt in CFG["textos_projecao"]}\n    (saida / "smoke_resultado.json").write_text(json.dumps(resumo, indent=2))\n    log("RESUMO", json.dumps({k: v for k, v in resumo.items() if k != "gpus"}))\n    for g in resumo["gpus"]:\n        log("melhor da gpu", g["info"].get("gpu"), json.dumps(g["melhor"]))\n\n\ndef shutil_which(nome):\n    import shutil\n\n    return shutil.which(nome)\n\n\nif __name__ == "__main__":\n    worker() if GPU_TAG is not None else launcher()\n'
caminho = Path("/kaggle/working/smoke_lora24.py")
caminho.parent.mkdir(parents=True, exist_ok=True)
caminho.write_text(SCRIPT)
processo = subprocess.Popen([sys.executable, str(caminho)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for linha in processo.stdout:
    print(linha, end="")
codigo = processo.wait()
print("código de saída:", codigo)
resultado = Path("/kaggle/working/smoke_resultado.json")
if resultado.exists():
    print(json.dumps(json.loads(resultado.read_text()), indent=2)[:6000])
assert codigo == 0
